In [1]:
import json
import pandas as pd
import os
import numpy as np
import requests
import argparse
from datetime import datetime, timezone
from datetime import timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pprint import pprint

import ipywidgets as widgets
from IPython.display import display, HTML


In [2]:
_config_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "stations.json")
with open(_config_path, encoding="utf-8") as _f:
    sat_dict = json.load(_f)


In [3]:
URL_BASE = "http://prodterh-fssaws:8080/FewsWebServices/rest/fewspiservice/v1/timeseries"

In [4]:
# obtener la fecha y hora de hoy en formato UTC
fecha_hoy = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
print(f"la fecha de hoy es: {fecha_hoy}")
print("----------------------------------")

# obtener 5 dias antes y 5 días después de la fecha de hoy en formato UTC
fecha_inicio = (datetime.now(timezone.utc) - timedelta(days=3)).strftime('%Y-%m-%dT%H:%M:%SZ')
print(f"la fecha de inicio es: {fecha_inicio}")

la fecha de hoy es: 2026-08-14T15:10:49Z
----------------------------------
la fecha de inicio es: 2026-08-11T15:10:49Z


In [5]:
data_dict = {}
with requests.Session() as session:
    for section in sat_dict:
        print(section)
        for location in sat_dict[section]:
                print(f"  Location: {location}")
                params = dict(
                    documentFormat="PI_JSON",
                    locationIds=sat_dict[section][location]['locationIds'],
                    parameterIds=sat_dict[section][location]['parameterIds'],
                    moduleInstanceIds=sat_dict[section][location]['moduleInstanceIds'],
                    startTime=fecha_inicio,
                    endTime=fecha_hoy
                    )

                response = session.get(url=URL_BASE, params=params)
                if response.status_code == 200:
                    data = response.json()
                else:
                    print(f"Error {response.status_code} for section {section}: {response.text}")
                    data_dict[section] = []
                    continue
                timeSeries = data.get('timeSeries', [])
                if not timeSeries:
                    print(f"No timeSeries data returned for section {section}")
                if section not in data_dict:
                    data_dict[section] = {}
                data_dict[section][location] = timeSeries
        print("==================================")


rio cuareim
  Location: Cuareim Rio
  Location: Catalan Grande
  Location: Artigas
rio yi
  Location: Sarandi del Yí
  Location: Polanco del Yí
  Location: Durazno
  Location: Mansavillagra R6
rio santa lucia
  Location: Fray Marcos
  Location: Paso Pache
  Location: Santa Lucia R11
  Location: Florida
rio san jose
  Location: Picada Varela
rio uruguay
  Location: Bella Unión
  Location: Salto
  Location: Paysandú
  Location: Fray Bentos
rio olimar grande
  Location: Treinta y Tres
rio negro
  Location: Mercedes
rio yaguaron
  Location: Passo das Pedras


In [6]:
df_dict = {}
for sec, loc_dict in data_dict.items():
    df_dict[sec] = {}
    for loc, ts_list in loc_dict.items():
        if not ts_list:
            df_dict[sec][loc] = pd.DataFrame(columns=['datetime', 'value', 'flag'])
            continue
        events = ts_list[0]['events']
        miss_val = float(ts_list[0]['header']['missVal'])
        df = pd.DataFrame(events)
        df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'])
        # convert from UTC to local time (UTC-3)
        df['datetime'] = df['datetime'] - timedelta(hours=3)
        df['value'] = pd.to_numeric(df['value']).replace(miss_val, float('nan'))
        df['flag'] = df['flag'].astype(int)
        df_dict[sec][loc] = df[['datetime', 'value', 'flag']].set_index('datetime')

# for the df_dict["rio uruguay"]["Monte Caseros"] the values should be substracted by 0.25 m 
df_dict["rio uruguay"]["Bella Unión"]['value'] = df_dict["rio uruguay"]["Bella Unión"]['value'] - 0.25


In [7]:
def get_trend(df, tol=0.01):
    vals = df['value'].dropna()
    if len(vals) < 2:
        return "–"

    last_val  = vals.iloc[-1]
    last_time = vals.index[-1]

    target_time = last_time - timedelta(hours=1)
    secs = (vals.index - target_time).total_seconds()
    idx_closest = int(np.abs(secs).argmin())
    prev_val = vals.iloc[idx_closest]


    diff = last_val - prev_val
    print(f"last_val: {last_val}, prev_val: {prev_val}, diff: {diff}")

    if diff > tol:
        return "▲"
    elif diff < -tol:
        return "▼"
    return "〓"

rows = []
for sec, locs in df_dict.items():
    for loc, df in locs.items():
        vals = df['value'].dropna()
        if vals.empty:
            continue
        trend = get_trend(df)
        nivel = vals.iloc[-1]
        last_dt = vals.index[-1].strftime('%d-%m-%Y %H:%M')
        umbrales = sat_dict.get(sec, {}).get(loc, {}).get('umbrales', {})
        avi = umbrales.get('aviso')
        seg = umbrales.get('seguridad')
        rows.append({
            'Cuenca':             sec.title(),
            'Estación':           loc,
            'Nivel (m)':          f"{nivel:.2f} {trend}",
            'Fecha':              last_dt,
            'Cota Aviso (m)':     f"{avi:.1f}" if avi is not None else "–",
            'Cota Seguridad (m)': f"{seg:.1f}" if seg is not None else "–",
        })

summary = pd.DataFrame(rows)

def color_nivel(row):
    styles = [''] * len(row)
    cols = list(row.index)
    nivel_idx = cols.index('Nivel (m)')
    try:
        nivel_val = float(row['Nivel (m)'].split()[0])
        seg = row['Cota Seguridad (m)']
        avi = row['Cota Aviso (m)']
        if seg != '–' and nivel_val >= float(seg):
            styles[nivel_idx] = 'background-color: #f28b82; color: white; font-weight: bold'
        elif avi != '–' and nivel_val >= float(avi):
            styles[nivel_idx] = 'background-color: #fdd663; color: black; font-weight: bold'
    except (ValueError, AttributeError):
        pass
    return styles


last_val: 1.06, prev_val: 1.06, diff: 0.0
last_val: 4.11, prev_val: 4.11, diff: 0.0
last_val: 2.32, prev_val: 2.32, diff: 0.0
last_val: 1.63, prev_val: 1.64, diff: -0.010000000000000009
last_val: 3.61, prev_val: 3.63, diff: -0.020000000000000018
last_val: 3.07, prev_val: 3.1, diff: -0.03000000000000025
last_val: 2.64, prev_val: 2.65, diff: -0.009999999999999787
last_val: 1.13, prev_val: 1.13, diff: 0.0
last_val: 0.86, prev_val: 0.86, diff: 0.0
last_val: 2.18, prev_val: 2.2, diff: -0.020000000000000018
last_val: 0.95, prev_val: 0.95, diff: 0.0
last_val: 4.57, prev_val: 4.59, diff: -0.019999999999999574
last_val: 8.72, prev_val: 8.75, diff: -0.02999999999999936
last_val: 4.36, prev_val: 4.36, diff: 0.0
last_val: 2.59, prev_val: 2.6, diff: -0.010000000000000231
last_val: 1.66, prev_val: 1.67, diff: -0.010000000000000009
last_val: 3.78, prev_val: 3.77, diff: 0.009999999999999787
last_val: 2.8, prev_val: 2.81, diff: -0.010000000000000231


In [8]:
cuencas = ['Todos'] + sorted(summary['Cuenca'].unique().tolist())

dropdown = widgets.Dropdown(
    options=cuencas,
    value='Todos',
    description='Cuenca:',
    layout=widgets.Layout(width='320px'),
    style={'description_width': '70px'},
)

output = widgets.Output()

def show_table(cuenca):
    with output:
        output.clear_output(wait=True)
        df_filt = summary if cuenca == 'Todos' else summary[summary['Cuenca'] == cuenca]
        display(
            df_filt.style
                .apply(color_nivel, axis=1)
                .set_properties(**{'text-align': 'center'})
                .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center'), ('background-color', '#104496'), ('color', 'white')]}])
                .hide(axis='index')
        )

dropdown.observe(lambda change: show_table(change['new']), names='value')
show_table('Todos')
display(dropdown, output)


Dropdown(description='Cuenca:', layout=Layout(width='320px'), options=('Todos', 'Rio Cuareim', 'Rio Negro', 'R…

Output()